# 01s — VBET Smoke Test (Red Shirt, 20 × 20 km)

A fast end-to-end run of the valley-bottom pipeline on a small box, so breakages show up in
seconds instead of half an hour. Same steps as the full tile run, just a smaller extent.

**Produces:** a valley-bottom polygon, a HAND raster, and a run manifest.

## 0. Setup

In [ ]:
import os, sys

# PROJ/GDAL paths must be set before any geospatial import: the Jupyter kernel starts
# without `conda activate`, so PROJ cannot otherwise find its database.
def _find_share(name):
    for base in (sys.prefix, sys.base_prefix):
        p = os.path.join(base, "share", name)
        if os.path.isdir(p):
            return p
    return None

_proj, _gdal = _find_share("proj"), _find_share("gdal")
if _proj:
    os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = _proj
if _gdal:
    os.environ.setdefault("GDAL_DATA", _gdal)

import json, math, time, shutil, subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
import rasterio.plot
from rasterio.features import shapes, rasterize
from shapely.geometry import box, shape
from shapely.ops import unary_union
from scipy.ndimage import label, binary_fill_holes
from osgeo import gdal
import whitebox


def _repo_root():
    """Walk up from the working directory to the repo root."""
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / ".git").exists() or (p / "environment.yml").is_file():
            return p
    raise RuntimeError("Could not find the repo root from " + str(here))

REPO     = _repo_root()
DATA_DIR = REPO / "data"
RUNS_DIR = REPO / "runs"          # manifests live here because data/ is gitignored
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"  repo : {REPO}")
print(f"  data : {DATA_DIR}")

## 1. Configuration

Everything you might change lives in this one cell.

In [ ]:
# ---- Where ----
# Centre of the smoke box: USGS 06403700, Cheyenne R at Red Shirt.
# This sits inside MGRS square 13TFJ, the pilot tile for the full run.
CENTRE_LON, CENTRE_LAT = -102.8921, 43.6724
BOX_HALF_M   = 10_000          # half-width -> a 20 x 20 km box
CRS_PROJ     = "EPSG:32613"    # UTM 13N

# ---- Grid ----
DEM_RES_M    = 30              # 3DEP 1 arc-second
DEM_PRODUCT  = "1"             # "1" = 1 arc-sec (~30 m), "13" = 1/3 arc-sec (~10 m)

# ---- Hydrology ----
# Least-cost breaching carves through blockages up to this far. 100 cells = 3 km at 30 m.
BREACH_DIST_CELLS = 100

# ---- VBET thresholds by drainage area class ----
# Each: drainage area window (km2), max height above stream (m), max slope (deg), buffer (m)
VBET_CLASSES = [
    {"label": "large",  "da_min": 1000, "da_max": 1e9,  "hand_m": 12, "slope_deg": 6,  "buffer_m": 1000},
    {"label": "medium", "da_min": 100,  "da_max": 1000, "hand_m": 8,  "slope_deg": 8,  "buffer_m": 500},
    {"label": "small",  "da_min": 0,    "da_max": 100,  "hand_m": 4,  "slope_deg": 12, "buffer_m": 200},
]
MIN_PATCH_HA = 1.0             # drop valley-bottom patches smaller than this

# Which size classes go into the analysis mask. The 162 small-class reaches in this box
# are ephemeral badlands draws: they contributed over half the valley-bottom area but
# are not cottonwood gallery habitat. Every reach is still classed and written out, so
# adding "small" back later is a one-word change, not a re-run of the hydrology.
ANALYSIS_CLASSES = ["large", "medium"]

# ---- Outputs ----
RUN_NAME    = "smoketest_redshirt"
OUT_DIR     = DATA_DIR / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_GPKG = OUT_DIR / f"{RUN_NAME}_valley_bottom.gpkg"

print(f"Box     : {2 * BOX_HALF_M / 1000:.0f} x {2 * BOX_HALF_M / 1000:.0f} km at "
      f"{CENTRE_LAT:.4f}, {CENTRE_LON:.4f}")
print(f"Grid    : {2 * BOX_HALF_M // DEM_RES_M} x {2 * BOX_HALF_M // DEM_RES_M} cells "
      f"({(2 * BOX_HALF_M / DEM_RES_M) ** 2 / 1e6:.2f} M) at {DEM_RES_M} m")
print(f"Outputs : {OUT_DIR}")

## 2. The box and its flowlines

In [ ]:
# Build the box in projected coordinates so BOX_HALF_M is genuinely metres.
centre = gpd.GeoSeries.from_xy([CENTRE_LON], [CENTRE_LAT], crs=4326).to_crs(CRS_PROJ).iloc[0]
box_proj = gpd.GeoDataFrame(
    geometry=[box(centre.x - BOX_HALF_M, centre.y - BOX_HALF_M,
                  centre.x + BOX_HALF_M, centre.y + BOX_HALF_M)],
    crs=CRS_PROJ,
)
box_wgs = box_proj.to_crs(4326)
BOX_BOUNDS_WGS = tuple(box_wgs.total_bounds)   # minx, miny, maxx, maxy

print(f"Box bounds (WGS84): {[round(v, 4) for v in BOX_BOUNDS_WGS]}")

In [ ]:
# Flowlines need a drainage area (totdasqkm) so each reach gets the right VBET class.
# Prefer the file 01a builds for the whole corridor; otherwise pull just this box from
# the NHDPlus v2 network service, which already carries the attribute.
VAA_GPKG = DATA_DIR / "cheyenne_flowlines_vaa.gpkg"

if VAA_GPKG.exists():
    flw = gpd.read_file(VAA_GPKG, layer="flowlines").to_crs(CRS_PROJ)
    flw = gpd.clip(flw, box_proj)
    FLOWLINE_SOURCE = VAA_GPKG.name
    print(f"Flowlines from {VAA_GPKG.name}: {len(flw):,} reaches in the box")
else:
    from pynhd import WaterData
    flw = (WaterData("nhdflowline_network")
           .bybox(BOX_BOUNDS_WGS)
           .to_crs(CRS_PROJ))
    flw = gpd.clip(flw, box_proj)
    FLOWLINE_SOURCE = "NHDPlus v2 nhdflowline_network (WaterData bybox)"
    print(f"Flowlines from NHDPlus v2 web service: {len(flw):,} reaches in the box")

flw = flw.rename(columns=str.lower).set_geometry("geometry")
assert "totdasqkm" in flw.columns, f"No drainage area column. Got: {list(flw.columns)}"
print(f"  drainage area: {flw.totdasqkm.min():.1f} - {flw.totdasqkm.max():,.0f} km2")
print(f"  channel length: {flw.geometry.length.sum() / 1000:,.1f} km")

In [ ]:
# ---- Assign each reach to a VBET size class ----
da = pd.to_numeric(flw["totdasqkm"], errors="coerce")
flw["vbet_class"] = np.select(
    [(da >= c["da_min"]) & (da < c["da_max"]) for c in VBET_CLASSES],
    [c["label"] for c in VBET_CLASSES],
    default="small",
)

print("Reaches per class:")
for c in VBET_CLASSES:
    sel = flw[flw.vbet_class == c["label"]]
    print(f"  {c['label']:6s} (HAND<{c['hand_m']:2d}m, slope<{c['slope_deg']:2d}deg, "
          f"buf {c['buffer_m']:4d}m): {len(sel):4,} reaches, "
          f"{sel.geometry.length.sum() / 1000:7,.1f} km")

In [ ]:
# Where the box sits in the corridor, and how the reaches classed.
CORRIDOR = DATA_DIR / "cheyenne_corridor_aoi.gpkg"
COLOURS = {"large": "#0D47A1", "medium": "#42A5F5", "small": "#B0BEC5"}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)

if CORRIDOR.exists():
    gpd.read_file(CORRIDOR, layer="study_area").to_crs(CRS_PROJ).plot(
        ax=a1, facecolor="#ECEFF1", edgecolor="#90A4AE", linewidth=0.6)
    gpd.read_file(CORRIDOR, layer="flowlines").to_crs(CRS_PROJ).plot(
        ax=a1, color="#42A5F5", linewidth=0.25)
box_proj.boundary.plot(ax=a1, color="#C62828", linewidth=2.0)
a1.set_title("Smoke-test box (red) in the Cheyenne corridor")

for lab, col in COLOURS.items():
    sel = flw[flw.vbet_class == lab]
    if len(sel):
        sel.plot(ax=a2, color=col, linewidth={"large": 2.4, "medium": 1.3, "small": 0.6}[lab],
                 label=f"{lab} ({len(sel)})")
box_proj.boundary.plot(ax=a2, color="#C62828", linewidth=1.2)
a2.legend(title="drainage-area class", loc="upper right", fontsize=9)
a2.set_title("Reaches by size class")

for ax in (a1, a2):
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 3. DEM

Static 3DEP tiles from the USGS S3 bucket — the same source the full run uses.

In [ ]:
TILE_DIR = DATA_DIR / "dem_tiles"
TILE_DIR.mkdir(parents=True, exist_ok=True)
BASE_URL = ("https://prd-tnm.s3.amazonaws.com/StagedProducts/Elevation/"
            "{p}/TIFF/current/{t}/USGS_{p}_{t}.tif")

def tiles_for_bbox(minx, miny, maxx, maxy):
    """1-degree 3DEP tile names covering a WGS84 bbox.

    Tiles are named by their NORTHWEST corner: n44w104 spans lat [43, 44], lon [-104, -103].
    """
    lats = range(math.floor(miny) + 1, math.ceil(maxy) + 1)
    lons = range(math.floor(-maxx) + 1, math.ceil(-minx) + 1)
    return [f"n{a:02d}w{o:03d}" for a in lats for o in lons]

TILES = tiles_for_bbox(*BOX_BOUNDS_WGS)
tile_paths, TILE_URLS = [], []

for t in TILES:
    url  = BASE_URL.format(p=DEM_PRODUCT, t=t)
    dest = TILE_DIR / f"USGS_{DEM_PRODUCT}_{t}.tif"
    TILE_URLS.append(url)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"  {dest.name}: cached ({dest.stat().st_size / 1e6:.0f} MB)")
    else:
        print(f"  {dest.name}: downloading…")
        r = subprocess.run(["curl", "-fL", "-C", "-", "--retry", "3", "-o", str(dest), url],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"Download failed for {t}: {r.stderr[-300:]}")
        print(f"    done ({dest.stat().st_size / 1e6:.0f} MB)")
    tile_paths.append(str(dest))

print(f"\n{len(tile_paths)} tile(s) covering the box")

In [ ]:
# Mosaic -> reproject -> clip to the box, in one GDAL pass.
DEM_PATH = OUT_DIR / f"{RUN_NAME}_dem_{DEM_RES_M}m.tif"
CUTLINE  = OUT_DIR / "_box_cutline.gpkg"
box_wgs[["geometry"]].to_file(CUTLINE, layer="cutline", driver="GPKG")

if DEM_PATH.exists():
    print(f"{DEM_PATH.name} exists — delete it to rebuild.")
else:
    vrt = str(OUT_DIR / "_dem_tiles.vrt")
    gdal.BuildVRT(vrt, tile_paths)
    gdal.Warp(
        str(DEM_PATH), vrt,
        dstSRS=CRS_PROJ, xRes=DEM_RES_M, yRes=DEM_RES_M, resampleAlg="bilinear",
        cutlineDSName=str(CUTLINE), cropToCutline=True,
        dstNodata=-9999.0, outputType=gdal.GDT_Float32,
        creationOptions=["COMPRESS=DEFLATE", "TILED=YES", "NUM_THREADS=ALL_CPUS"],
    )
    print(f"  wrote {DEM_PATH.name} ({DEM_PATH.stat().st_size / 1e6:.1f} MB)")

with rasterio.open(DEM_PATH) as src:
    dem_shape, dem_transform, dem_profile = src.shape, src.transform, src.profile.copy()
    dem = src.read(1, masked=True)

print(f"  {dem_shape[1]} x {dem_shape[0]} cells | "
      f"elevation {dem.min():.0f} - {dem.max():.0f} m")

## 4. WhiteboxTools

In [ ]:
# WBT_PATH must be set BEFORE constructing WhiteboxTools: the constructor calls
# download_wbt(), which returns early when that variable is set. Without it, CyVerse
# re-downloads the ~200 MB binary every session because /opt/conda does not persist.
WBT_PERSIST_DIR = Path.home() / "data-store" / "bin" / "WBT"
WBT_EXE = "whitebox_tools.exe" if sys.platform.startswith("win") else "whitebox_tools"
have_persisted = (WBT_PERSIST_DIR / WBT_EXE).exists()

if have_persisted:
    os.environ["WBT_PATH"] = str(WBT_PERSIST_DIR)
    print(f"Using persistent WhiteboxTools at {WBT_PERSIST_DIR}")
else:
    print("No persisted binary — WhiteboxTools downloads ~200 MB on first use.")

wbt = whitebox.WhiteboxTools()
wbt.verbose = False
if have_persisted:
    wbt.set_whitebox_dir(str(WBT_PERSIST_DIR))
wbt.set_max_procs(-1)                        # -1 = all cores
wbt.set_working_dir(str(OUT_DIR.resolve()))  # WBT resolves bare filenames against this

WBT_VERSION = wbt.version().splitlines()[0].strip()
print(f"{WBT_VERSION}")

## 5. Hydrology

Height Above Nearest Drainage (HAND) is measured against the **NHD flowlines**, not against a
stream network derived from flow accumulation.

Flow accumulation is the one step that needs the whole upstream watershed, so it cannot be
computed correctly inside a clipped box — every stream entering from outside would start at
zero. Using NHD instead removes that dependency, and NHD already carries surveyed drainage
areas. What remains is three tool calls.

In [ ]:
# ---- Rasterize the NHD flowlines as the stream network ----
# Background must be 0, not nodata, or HAND will not propagate away from the channel.
streams_path = OUT_DIR / f"{RUN_NAME}_streams_{DEM_RES_M}m.tif"

stream_arr = rasterize(
    ((geom, 1) for geom in flw.geometry),
    out_shape=dem_shape, transform=dem_transform,
    fill=0, dtype="float32", all_touched=True,
)

prof = dem_profile.copy()
prof.update(dtype="float32", count=1, nodata=-9999.0, compress="deflate")
with rasterio.open(streams_path, "w", **prof) as dst:
    dst.write(stream_arr, 1)

print(f"Stream cells: {int(stream_arr.sum()):,} "
      f"({int(stream_arr.sum()) * DEM_RES_M / 1000:,.0f} km of channel)")

In [ ]:
# ---- The chain: breach -> HAND -> slope ----
breached_path = OUT_DIR / f"{RUN_NAME}_dem_breached_{DEM_RES_M}m.tif"
hand_path     = OUT_DIR / f"{RUN_NAME}_hand_{DEM_RES_M}m.tif"
slope_path    = OUT_DIR / f"{RUN_NAME}_slope_{DEM_RES_M}m.tif"

n = lambda p: p.name   # WBT resolves bare filenames against its working directory

def run_step(name, out_path, fn):
    """Run one WBT step unless its output is already on disk."""
    if out_path.exists():
        print(f"  [cached] {name}")
        return
    t0 = time.time()
    print(f"  [run   ] {name} …", end="", flush=True)
    rc = fn()
    if rc != 0 or not out_path.exists():
        raise RuntimeError(f"WhiteboxTools step '{name}' failed (exit {rc}). "
                           f"Set wbt.verbose = True to see the tool output.")
    print(f" {time.time() - t0:.0f} s")

t_start = time.time()
run_step("BreachDepressionsLeastCost", breached_path, lambda: wbt.breach_depressions_least_cost(
    n(DEM_PATH), n(breached_path), dist=BREACH_DIST_CELLS, fill=True))

run_step("ElevationAboveStream (HAND)", hand_path, lambda: wbt.elevation_above_stream(
    n(breached_path), n(streams_path), n(hand_path)))

run_step("Slope", slope_path, lambda: wbt.slope(
    n(breached_path), n(slope_path), units="degrees"))

HYDRO_SECONDS = time.time() - t_start
print(f"\nHydrology done in {HYDRO_SECONDS:.0f} s")

## 6. Check the flowlines against the terrain

HAND is measured from the NHD lines, so if those lines sit off the real channel the whole
result shifts with them. Compare the elevation under the flowlines to the lowest ground nearby:
the difference should be small.

In [ ]:
with rasterio.open(hand_path) as src:
    hand = src.read(1, masked=True)
with rasterio.open(slope_path) as src:
    slope = src.read(1, masked=True)

# Elevation under each stream cell vs. the minimum elevation within ~5 cells of it.
from scipy.ndimage import minimum_filter
stream_mask = stream_arr > 0
local_min   = minimum_filter(np.ma.filled(dem, np.inf), size=11)
offset      = np.ma.filled(dem, np.nan)[stream_mask] - local_min[stream_mask]
offset      = offset[np.isfinite(offset)]   # drop stream cells that sat on nodata

print(f"Elevation above the local minimum, under the flowlines:")
print(f"  median {np.nanmedian(offset):5.1f} m | 90th pct {np.nanpercentile(offset, 90):5.1f} m "
      f"| max {np.nanmax(offset):5.1f} m")
print("  A median of a few metres is normal. Tens of metres means the flowlines are")
print("  off the channel and HAND should not be trusted until they are snapped.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), constrained_layout=True)
with rasterio.open(DEM_PATH) as _src:
    ext = rasterio.plot.plotting_extent(_src)

for ax, arr, title, cmap, vmax in [
    (axes[0], dem,   "Elevation (m)",     "terrain", None),
    (axes[1], hand,  "HAND (m)",          "viridis_r", 30),
    (axes[2], slope, "Slope (degrees)",   "magma",   25),
]:
    im = ax.imshow(arr, cmap=cmap, extent=ext, origin="upper", vmax=vmax)
    flw.plot(ax=ax, color="#1565C0", linewidth=0.8)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, shrink=0.72)

fig.suptitle("Smoke test — NHD flowlines in blue", fontsize=13)
plt.show()

In [ ]:
# Why the thresholds matter: most of the landscape sits well above the stream.
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(hand.compressed(), bins=120, range=(0, 60), color="#78909C")
for c in VBET_CLASSES:
    ax.axvline(c["hand_m"], color=COLOURS[c["label"]], linewidth=2,
               label=f"{c['label']}: HAND < {c['hand_m']} m")
ax.set_xlabel("Height above nearest drainage (m)")
ax.set_ylabel("cells")
ax.set_title("HAND distribution and the class cut-offs")
ax.legend()
plt.show()

## 7. Valley bottom

In [ ]:
# A cell is valley bottom if it is near a reach, low above the stream, and not steep.
# Thresholds come from that reach's size class.
valid = ~np.ma.getmaskarray(hand) & ~np.ma.getmaskarray(slope)
hand_arr, slope_arr = np.ma.filled(hand, 9999.0), np.ma.filled(slope, 90.0)
valley_mask = np.zeros(dem_shape, dtype=np.uint8)

for c in VBET_CLASSES:
    sel = flw[flw.vbet_class == c["label"]]
    if c["label"] not in ANALYSIS_CLASSES:
        excl = sel.geometry.length.sum() / 1000
        print(f"  {c['label']:6s}: {len(sel):4,} reaches ({excl:,.0f} km) excluded from the mask")
        continue
    if len(sel) == 0:
        print(f"  {c['label']:6s}: no reaches")
        continue

    # Rasterizing overlapping buffers to the same value already unions them —
    # no need to union the geometries first.
    near = rasterize(((g, 1) for g in sel.geometry.buffer(c["buffer_m"])),
                     out_shape=dem_shape, transform=dem_transform,
                     fill=0, dtype=np.uint8, all_touched=True)

    cls_mask = (near == 1) & (hand_arr < c["hand_m"]) & (slope_arr < c["slope_deg"]) & valid
    np.maximum(valley_mask, cls_mask, out=valley_mask, casting="unsafe")
    print(f"  {c['label']:6s}: {int(cls_mask.sum()):8,} cells "
          f"({int(cls_mask.sum()) * DEM_RES_M ** 2 / 1e6:6.1f} km2)")

print(f"\nCombined: {int(valley_mask.sum()):,} cells "
      f"({int(valley_mask.sum()) * DEM_RES_M ** 2 / 1e6:.1f} km2)")

In [ ]:
# ---- Clean up and vectorize ----
valley = binary_fill_holes(valley_mask).astype(np.uint8)   # fill enclosed upland pockets

labeled, n_found = label(valley)
min_cells = int((MIN_PATCH_HA * 1e4) / DEM_RES_M ** 2)
counts = np.bincount(labeled.ravel())
counts[0] = 0                                              # label 0 is background
keep = np.flatnonzero(counts >= min_cells)
valley = np.isin(labeled, keep).astype(np.uint8)

polys = [shape(g) for g, v in shapes(valley, mask=valley.astype(bool),
                                     transform=dem_transform) if v == 1]
valley_poly = unary_union(polys).buffer(DEM_RES_M * 2).buffer(-DEM_RES_M * 2)  # smooth edges

# Keep the patches as separate features rather than one dissolved blob. Later steps
# screen and summarise patch by patch, which a single merged polygon cannot support.
parts = list(getattr(valley_poly, "geoms", [valley_poly]))
patches = gpd.GeoDataFrame(geometry=parts, crs=CRS_PROJ)
patches = patches[patches.area > 0].reset_index(drop=True)
patches.insert(0, "patch_id", patches.index + 1)
patches["area_ha"] = patches.area / 1e4
VALLEY_KM2 = patches.area.sum() / 1e6

print(f"Components: {n_found:,} -> {len(keep):,} (kept >= {MIN_PATCH_HA} ha)")
print(f"Patches written: {len(patches):,}  "
      f"(largest {patches.area_ha.max():,.0f} ha, median {patches.area_ha.median():.1f} ha)")
print(f"Valley bottom: {VALLEY_KM2:.1f} km2 "
      f"({100 * VALLEY_KM2 / (box_proj.area.sum() / 1e6):.1f}% of the box)")

## 8. Save and record the run

The manifest is the small text file that makes this run reproducible: what went in, which
parameters, which code. It goes in `runs/` and is committed to git. The rasters are not.

In [ ]:
# ---- Binary mask raster, for clipping imagery later ----
# 1 = valley bottom, 0 = not. Same grid as the DEM, so it lines up with anything
# else built on this box without resampling.
MASK_PATH = OUT_DIR / f"{RUN_NAME}_valley_mask_{DEM_RES_M}m.tif"

# Burn the SAME smoothed polygon that gets written to the GeoPackage, so the raster
# and the vector describe the same area. Using the pre-smoothing array here instead
# leaves the two about 6% apart, which is the sort of mismatch nobody notices until
# an area statistic disagrees with a map.
valley_mask_out = rasterize(((g, 1) for g in patches.geometry), out_shape=dem_shape,
                            transform=dem_transform, fill=0, dtype="uint8")

prof = dem_profile.copy()
prof.update(dtype="uint8", count=1, nodata=255, compress="deflate")
with rasterio.open(MASK_PATH, "w", **prof) as dst:
    dst.write(valley_mask_out, 1)
    dst.write_colormap(1, {0: (245, 245, 245), 1: (46, 125, 50)})

burned_km2 = valley_mask_out.sum() * DEM_RES_M ** 2 / 1e6
print(f"Mask -> {MASK_PATH.name}  ({valley_mask_out.sum():,} cells = {burned_km2:.1f} km2, "
      f"polygon {VALLEY_KM2:.1f} km2)")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 7), constrained_layout=True)
a1.imshow(valley_mask_out, cmap="Greens", extent=ext, origin="upper", vmin=0, vmax=1)
a1.set_title("Binary mask (1 = valley bottom)")
a2.imshow(np.ma.masked_where(hand > 40, hand), cmap="Greys_r", extent=ext, origin="upper")
gpd.GeoSeries([valley_poly], crs=CRS_PROJ).plot(
    ax=a2, facecolor="#2E7D32", edgecolor="#1B5E20", alpha=0.45, linewidth=0.7)
flw.plot(ax=a2, color="#1565C0", linewidth=1.0)
a2.set_title(f"Valley bottom over terrain — {VALLEY_KM2:.1f} km2")
for ax in (a1, a2):
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

In [ ]:
def git_commit():
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO,
                              capture_output=True, text=True).stdout.strip() or None
    except Exception:
        return None

patches["method"]   = "VBET-simplified (NHD streams)"
patches["run_name"]  = RUN_NAME
patches["dem_res_m"] = DEM_RES_M
patches.to_file(OUTPUT_GPKG, layer="valley_bottom", driver="GPKG")
flw.to_file(OUTPUT_GPKG, layer="flowlines_classed", driver="GPKG")
print(f"Valley bottom -> {OUTPUT_GPKG.relative_to(REPO)}")

manifest = {
    "run_name":    RUN_NAME,
    "notebook":    "01s_VBET_SmokeTest.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git_commit":  git_commit(),
    "inputs": {
        "dem_tiles":       TILE_URLS,
        "dem_product":     DEM_PRODUCT,
        "flowline_source": FLOWLINE_SOURCE,
        "n_reaches":       int(len(flw)),
    },
    "parameters": {
        "centre_lon_lat":    [CENTRE_LON, CENTRE_LAT],
        "box_half_m":        BOX_HALF_M,
        "crs":               CRS_PROJ,
        "dem_res_m":         DEM_RES_M,
        "breach_dist_cells": BREACH_DIST_CELLS,
        "vbet_classes":      VBET_CLASSES,
        "min_patch_ha":      MIN_PATCH_HA,
        "analysis_classes":  ANALYSIS_CLASSES,
    },
    "environment": {
        "python":           sys.version.split()[0],
        "whiteboxtools":    WBT_VERSION,
        "geopandas":        gpd.__version__,
        "rasterio":         rasterio.__version__,
    },
    "results": {
        "valley_bottom_km2":   round(VALLEY_KM2, 3),
        "n_patches":           int(len(patches)),
        "box_km2":             round(box_proj.area.sum() / 1e6, 1),
        "hydrology_seconds":   round(HYDRO_SECONDS, 1),
        "flowline_offset_median_m": round(float(np.nanmedian(offset)), 2),
    },
    "outputs": [OUTPUT_GPKG.name, MASK_PATH.name, hand_path.name, slope_path.name],
}

manifest_path = RUNS_DIR / f"{RUN_NAME}.manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(f"Manifest      -> {manifest_path.relative_to(REPO)}")
print(json.dumps(manifest["results"], indent=2))